In [6]:
# import packages
from Equipments import BNC575, Weeder, SR400, MCBOX, Andor, Agilent, ThorlabsPM
from Equipments.MCBOX import generate_test_voltages
from DataProcessing.Plot import PlotSaver
import numpy as np
import pyvisa
import time
import matplotlib.pyplot as plt
from scipy.signal import find_peaks
from scipy.optimize import curve_fit

In [3]:
import matplotlib
import numpy as np

### Establish communication with instrumentation

In [4]:
ps = PlotSaver("X:/migratedData/Rydberg_QIS/data")

In [5]:
rm = pyvisa.ResourceManager()
print(rm.list_resources())

('ASRL1::INSTR', 'ASRL2::INSTR', 'ASRL3::INSTR', 'ASRL4::INSTR', 'ASRL5::INSTR', 'ASRL6::INSTR', 'ASRL7::INSTR', 'GPIB0::9::INSTR', 'TCPIP0::169.254.110.10::inst0::INSTR', 'USB0::0x1313::0x807B::241023118::0::INSTR')


In [11]:
bnc575 = BNC575.BNC575("GPIB0::9::INSTR")

In [ ]:
sr400 = SR400.SR400("GPIB0::23::INSTR", timeout = 1000)

In [ ]:
weeder = Weeder.Weeder("ASRL1::INSTR")

In [ ]:
mcbox = MCBOX.MCBOX(find_device = "USB-3114")

### Define the pulse sequence

In [26]:
pulse_arrangement = [["A", 5, 0, 2.3]
                     # ["B", 5, 0, 2],         # 780 EXC (10 dB attenuator connected) 
                    ]
T = 10
bnc575.clock_set(period = T, mode = "CONTINUOUS")
bnc575.pulse_sequence_setup(pulse_arrangement)
bnc575.start_pulses()

------------------------------------------------------------
The global clock is set to:
Mode: NORM
Period: 1.00e+01s
Pulses number: N/A
------------------------------------------------------------
[['A', 5, 0, 2.3]]
------------------------------------------------------------
The channel A is set to:
Width: 5.0000e+00s
Delay: 0.0000e+00s
Synchronize to: T0
Out: 2.30

Output timers:
HGFEDCBA
00000001
------------------------------------------------------------


In [ ]:
pulse_arrangement = [["A", 900e-06, 0, "TTL"],             # MOT (10 dB attenuator connected)
                     [["B", 10e-06, 5e-06, "TTL"],         # 780 EXC (10 dB attenuator connected) 
                      ["C", 10e-06, 5e-06, "TTL"]],        # 480 EXC
                     ["D", 10e-06, 0, "TTL"],              # u-wave switch (A*p drive)
                     ["E", 20e-06, 1e-06, 8]               # DEI switch/photon counter (may need to widen the pulse width to get both Rydberg states)
                    ]
# NOTES (18.03.26): 
# I swapped the cables connected to channels D and E on the BNC box to match the above pulse arrangement.
# According to the datasheet for the u-wave switch, when the TTL is on, the RF input should come out of port 2.

T = 1000e-6 # we may have to change this so the total time of the pulse sequence is <= T

cycle_number = 30

# photon counter gate width and delay (we'll need to adjust these as needed)
# GATE A: full gate (58 S 1/2 and 59 S 1/2), GATE B: partial gate (59 S 1/2 only)
gate_A_width = 15e-6
gate_A_delay = 3e-6

gate_B_width = 8e-06
gate_B_delay = 6e-06

# stepper motor slope (for blue laser)
freq_factor = 2 * 129 # kHz

notes = {"gate_A_width": gate_A_width, "gate_A_delay": gate_A_delay, "gate_B_width": gate_B_width, "gate_B_delay": gate_B_delay}

In [ ]:
sr400.counter_set(count_mode = "INDEPENDENT", count_preset = 100e-6, count_period_num = cycle_number, 
                  dwell = 0, sourceA = "INPUT1", sourceB = "INPUT1", 
                  gate_A_mode = "FIXED", gate_A_delay = gate_A_delay, gate_A_width = gate_A_width, 
                  gate_B_mode = "FIXED", gate_B_delay = gate_B_delay, gate_B_width = gate_B_width)

In [7]:
notes["bnc575 T0"] = bnc575.clock_set(period = T, mode = "CONTINUOUS")
notes["bnc575 pulses"] = bnc575.pulse_sequence_setup(pulse_arrangement)

------------------------------------------------------------
The global clock is set to:
Mode: NORM
Period: 1.00e-04s
Pulses number: N/A
------------------------------------------------------------


NameError: name 'notes' is not defined

In [11]:
bnc575.start_pulses()

### Search for a Rydberg resonance

In [ ]:
weeder.position(header = "A", query = True)

In [ ]:
# begin stepper motor at initial position 1000, then move it out to increase the frequency
weeder.move("A", position = 2800, progress = True)

In [ ]:
# used to move motor incrementally to zero in on the location of a resonance
weeder.advance(header = "A", num_step = 1)

### Microwave spectroscopy

In [ ]:
center = 2900 # center point of frequency scan
stride = 2 # step size
half_width = 200 # step range

# scan list generation
x_list = np.arange(2* half_width/stride)

# compensation voltage setpoints
vset = np.array([0.2,-0.1,0.6])

# MC channels for each electrode pair (+/- x, +/- y, +/- z)
electrode_pair = [[10,11],[6,7],[0,1]]

# set electrode voltages
mcbox.set_voltage_1chan(board_num = 0, channel = 10, voltage = vset[0], bipolar = True, display = True)
mcbox.set_voltage_1chan(board_num = 0, channel = 11, voltage = -vset[0], bipolar = True, display = True)

mcbox.set_voltage_1chan(board_num = 0, channel = 6, voltage = vset[1], bipolar = True, display = True)
mcbox.set_voltage_1chan(board_num = 0, channel = 7, voltage = -vset[1], bipolar = True, display = True)

mcbox.set_voltage_1chan(board_num = 0, channel = 0, voltage = 0, bipolar = True, display = True)
mcbox.set_voltage_1chan(board_num = 0, channel = 1, voltage = -vset[2], bipolar = True, display = True)

In [ ]:
# everytime go further and go back to release the band tension and gear margin
start_position = center - half_width
weeder.move(header = "A", position = start_position - 100)
weeder.move(header = "A", position = start_position)

results_gateA = []
results_gateB = []

s = time.perf_counter()
for idx in x_list:
    print(f"Now checking step: {idx}", end = "\r", flush = True)
    sr400.count_reset()
    result_gateA = np.mean(sr400.read_entire_counts(counter = "A", length = cycle_number))
    result_gateB = np.mean(sr400.read_entire_counts(counter = "B", length = cycle_number))
    results_gateA.append(result_gateA)
    results_gateB.append(result_gateB)
    for i in range(stride):
        weeder.step("A", "+")
    time.sleep(0.1)
e = time.perf_counter()
print('Time elapsed: {} seconds'.format(e-s))

results_prob = [x / y for x, y in zip(results_gateB, results_gateA)]

In [ ]:
# plot results
ps.Plot2D(x = x_list*stride*freq_factor, y = None, Z = results_gateA,
          xlabel = "480 laser frequency (kHz)", zlabel = "Average number of counts [GATE A]", notes = notes)

In [ ]:
# plot results
ps.Plot2D(x = x_list*stride*freq_factor, y = None, Z = results_gateB,
          xlabel = "480 laser frequency (kHz)", zlabel = "Average number of counts [GATE B]", notes = notes)

In [ ]:
# plot results
ps.Plot2D(x = x_list*stride*freq_factor, y = None, Z = results_prob,
          xlabel = "480 laser frequency (kHz)", zlabel = "58 $S_{1/2}$ - 59$S_{1/2}$ transition probability", notes = notes)

### Shutting down instrumention

In [ ]:
# reset compensation voltages to setpoint values
mcbox.set_voltage_Nchan(board_num = 0, channels = [0,1,6,7,10,11], voltages = [0,-vset[2],vset[1],-vset[1],vset[0],-vset[0]], bipolar = True)

In [ ]:
# save current stepper motor position
weeder.save(header = "A", query = True)

# disconnect from stepper motor?
weeder.pyvisa.close()

In [ ]:
# disconnect from BNC box
bnc575.pyvisa.close()

In [ ]:
# disconnect from SRS photon counter
sr400.pyvisa.close()

In [ ]:
# do a final check for any remaining open devices
rm.list_opened_resources()

In [ ]:
# close PyVISA resource manager
rm.close()